# NB04: Interpretation and discovery

SHAP analysis, PDP plots, hypothesis H3/H4/H5 tests, regulatory threshold metrics,
and open-ended discovery analyses.

**Outputs**
- `data/shap_importance.csv` — mean |SHAP| per feature per target
- `data/threshold_metrics.csv` — regulatory threshold sensitivity/specificity
- Figures in `figures/`

In [ ]:
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

for _cand in [Path.cwd() / 'scripts', Path.cwd().parent / 'scripts']:
    if _cand.exists():
        sys.path.insert(0, str(_cand))
        break

DATA_DIR = next(p for p in [Path.cwd() / 'data', Path.cwd().parent / 'data'] if p.exists())
FIGURES_DIR = next(p for p in [Path.cwd() / 'figures', Path.cwd().parent / 'figures'] if p.exists())
FIGURES_DIR.mkdir(exist_ok=True)

feature_matrix = pd.read_parquet(DATA_DIR / 'feature_matrix.parquet')
oof_preds = pd.read_parquet(DATA_DIR / 'oof_predictions.parquet')
cv_results = pd.read_csv(DATA_DIR / 'cv_results.csv')
boot_df = pd.read_csv(DATA_DIR / 'bootstrap_delta_rmse.csv')

targets = ['log_Cu_ppm', 'log_Zn_ppm', 'log_Pb_ppm', 'log_Ni_ppm']
targets_avail = [t for t in targets if t in feature_matrix.columns]

print('Loaded all data')
print(f'OOF predictions columns: {list(oof_preds.columns)}')

## 1. Refit M2 on full data for SHAP

In [ ]:
from modelling import build_xgboost, get_features, _drop_nan_rows

shap_models = {}
X_m2_full = get_features(feature_matrix, 'M2')

for target in targets_avail:
    y = feature_matrix[target]
    Xc, yc = _drop_nan_rows(X_m2_full, y)
    m = build_xgboost().fit(Xc, yc)
    shap_models[target] = (m, Xc)
    print(f'  Fit M2 for SHAP: {target} (n={len(Xc)})')

## 2. SHAP analysis

In [ ]:
from evaluation import compute_shap_values, shap_summary, shap_feature_importance_table

shap_values_dict = {}
shap_importance_parts = {}

for target in targets_avail:
    model, Xc = shap_models[target]
    sv = compute_shap_values(model, Xc, model_type='tree')
    shap_values_dict[target] = sv

    imp = shap_summary(
        sv, Xc,
        target_name=target.replace('log_', ''),
        top_n=20,
        save_path=FIGURES_DIR / f'shap_summary_{target}.png',
    )
    shap_importance_parts[target] = imp['mean_abs_shap']
    print(f'SHAP done: {target}')

# Combined importance table
shap_table = pd.DataFrame(shap_importance_parts)
shap_table['mean_across_targets'] = shap_table.mean(axis=1)
shap_table = shap_table.sort_values('mean_across_targets', ascending=False)
shap_table.to_csv(DATA_DIR / 'shap_importance.csv')
print(shap_table.head(15))

## 3. H3: Cofactor vs resistance CWM importance

In [ ]:
print('H3: Cofactor (metabolism) vs resistance (defense) CWM SHAP rank comparison')
print('='*60)

COFACTOR_COL = 'CWM_mean_n_metabolism_clusters'
RESISTANCE_COL = 'CWM_mean_n_defense_clusters'

# Check whether the metabolism CWM feature is degenerate (constant zero) in the
# genus trait table — the upstream source for this column.  If it is, the SHAP
# importance will be zero by construction and rank comparison is meaningless.
trait_table = pd.read_csv(DATA_DIR / 'genus_trait_table.csv')
met_col = 'mean_n_metabolism_clusters'
if met_col in trait_table.columns:
    met_vals = trait_table[met_col].dropna()
    met_is_constant_zero = len(met_vals) > 0 and float(met_vals.max()) == 0.0
else:
    met_is_constant_zero = True
    met_vals = pd.Series([], dtype=float)

if met_is_constant_zero:
    print(f'\nH3: UNTESTABLE')
    print(f'Reason: {met_col} in genus_trait_table.csv is 0.0 for all '
          f'{len(met_vals):,} non-NaN genera (max={float(met_vals.max()) if len(met_vals) else "N/A":.1f}, '
          f'n_nonzero={(met_vals > 0).sum()}).')
    print(f'The metabolism cluster column was never populated in the upstream '
          f'comprehensive_metal_ecology pangenome pipeline.')
    print()
    print(f'Because CWM_mean_n_metabolism_clusters = 0 for every sample in the '
          f'training data, its SHAP importance is zero by construction.  '
          f'The rank of 18/18 in the SHAP table (ranks 16–18 of 18 features) '
          f'reflects the constant-zero input, not a genuine biological test of '
          f'whether cofactor metabolism genes are more predictive than resistance genes.')
    print()
    # Show resistance CWM is a valid (non-degenerate) comparator
    def_col = 'mean_n_defense_clusters'
    if def_col in trait_table.columns:
        def_vals = trait_table[def_col].dropna()
        print(f'Defense CWM (valid comparator): n={len(def_vals):,}, '
              f'mean={def_vals.mean():.2f}, std={def_vals.std():.2f}, '
              f'n_nonzero={(def_vals > 0).sum():,}/{len(def_vals):,}')
    print()
    print('H3 verdict: UNTESTABLE')
    print('Fix: populate mean_n_metabolism_clusters in the comprehensive_metal_ecology '
          'pangenome pipeline (NB01), regenerate genus_trait_table.csv, and rebuild '
          'the feature matrix (NB00) before re-running this hypothesis test.')
else:
    # Normal path — metabolism column is populated; run the SHAP rank test
    h3_records = []
    for target in targets_avail:
        if target not in shap_table.columns:
            continue
        col_shap = shap_table[target].dropna().sort_values(ascending=False)
        ranked = pd.Series(range(1, len(col_shap) + 1), index=col_shap.index)

        cofactor_rank = ranked.get(COFACTOR_COL, np.nan)
        resistance_rank = ranked.get(RESISTANCE_COL, np.nan)

        h3_records.append({
            'target': target,
            'CWM_metabolism_rank': cofactor_rank,
            'CWM_defense_rank': resistance_rank,
            'metabolism_higher': (
                not np.isnan(float(cofactor_rank))
                and not np.isnan(float(resistance_rank))
                and cofactor_rank < resistance_rank
            ),
        })
        print(f'{target}: metabolism(cofactor) rank={cofactor_rank}, '
              f'defense(resistance) rank={resistance_rank}')

    h3_df = pd.DataFrame(h3_records)
    n_support = h3_df['metabolism_higher'].sum()
    print(f'\nH3 supported for {n_support}/{len(h3_df)} metals '
          f'(success criterion: ≥3)')
    print('H3:', 'SUPPORTED' if n_support >= 3 else 'NOT SUPPORTED')


## 4. H4: ΔRMSE rank by metal (Cu and Ni largest)

In [ ]:
print('H4: ΔRMSE (M4 − M2) rank by target metal')
h4_df = boot_df[boot_df['hypothesis'] == 'H2'][['target', 'observed_delta_rmse']].copy()
h4_df['rank'] = h4_df['observed_delta_rmse'].rank(ascending=False).astype(int)
print(h4_df.sort_values('rank').to_string(index=False))

cu_rank = h4_df.loc[h4_df['target'] == 'log_Cu_ppm', 'rank'].values
ni_rank = h4_df.loc[h4_df['target'] == 'log_Ni_ppm', 'rank'].values
h4_supported = all(r <= 2 for r in [*cu_rank, *ni_rank] if not np.isnan(r))
print(f'\nH4: {"SUPPORTED" if h4_supported else "NOT SUPPORTED"} '
      f'(Cu rank={cu_rank}, Ni rank={ni_rank}; need both ≤ 2)')

## 5. PDP for top features

In [ ]:
from evaluation import pdp_plot

top_features = shap_table.head(5).index.tolist()
target = targets_avail[0]  # Cu as example
model, Xc = shap_models[target]

for feat in top_features:
    if feat in Xc.columns:
        pdp_plot(
            model, Xc, feature=feat,
            target_name=target.replace('log_', ''),
            save_path=FIGURES_DIR / f'pdp_{target}_{feat}.png',
        )
print('PDP plots saved.')

## 6. Regulatory threshold metrics

In [ ]:
from evaluation import threshold_metrics

thresh_records = []
for target in targets_avail:
    raw_target = target.replace('log_', '')  # e.g., 'Cu_ppm'
    y = feature_matrix[target]
    oof_col = f'M2_{target}'
    if oof_col not in oof_preds.columns:
        continue
    y_pred = oof_preds[oof_col]
    valid = y.notna() & y_pred.notna()
    if valid.sum() < 10:
        continue
    rec = threshold_metrics(y[valid], y_pred[valid].values, raw_target, log_transformed=True)
    thresh_records.append(rec)

thresh_df = pd.DataFrame(thresh_records)
if not thresh_df.empty:
    thresh_df.to_csv(DATA_DIR / 'threshold_metrics.csv', index=False)
    print(thresh_df[['target', 'threshold_ppm', 'n_above', 'sensitivity', 'specificity', 'ppv']].to_string(index=False))

## 7. Discovery: spatial residual map

In [ ]:
target = targets_avail[0]
y = feature_matrix[target]
oof_m2_col = f'M2_{target}'

if oof_m2_col in oof_preds.columns:
    oof_m2 = oof_preds[oof_m2_col]
    residuals = y - oof_m2
    valid = residuals.notna() & feature_matrix['lat'].notna()

    fig, ax = plt.subplots(figsize=(10, 5))
    sc = ax.scatter(
        feature_matrix.loc[valid, 'lon'],
        feature_matrix.loc[valid, 'lat'],
        c=residuals[valid], cmap='RdBu_r', s=8, alpha=0.6,
        vmin=-2, vmax=2,
    )
    plt.colorbar(sc, ax=ax, label='Residual (observed − predicted)')
    ax.set_title(f'Spatial residuals: M2 for {target.replace("log_", "")}')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f'residual_map_{target}.png', dpi=150)
    plt.close()
    print('Spatial residual map saved.')

## 8. Discovery: CWM coverage vs prediction error

In [ ]:
target = targets_avail[0]
y = feature_matrix[target]
oof_m2_col = f'M2_{target}'
coverage = feature_matrix.get('coverage_fraction')

if oof_m2_col in oof_preds.columns and coverage is not None:
    oof_m2 = oof_preds[oof_m2_col]
    abs_resid = (y - oof_m2).abs()
    valid = abs_resid.notna() & coverage.notna()

    fig, ax = plt.subplots(figsize=(5, 4))
    ax.scatter(coverage[valid], abs_resid[valid], alpha=0.3, s=8)
    r = coverage[valid].corr(abs_resid[valid], method='spearman')
    ax.set_xlabel('CWM coverage fraction')
    ax.set_ylabel('|Residual|')
    ax.set_title(f'Coverage vs error ({target.replace("log_", "")}): ρ={r:.3f}')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f'coverage_vs_error_{target}.png', dpi=150)
    plt.close()
    print(f'Spearman ρ (coverage vs |residual|) = {r:.3f}')

## 9. Discovery: Metal co-prediction analysis

In [ ]:
resids = {}
for target in targets_avail:
    y = feature_matrix[target]
    oof_m2_col = f'M2_{target}'
    if oof_m2_col in oof_preds.columns:
        resids[target] = y - oof_preds[oof_m2_col]

if len(resids) >= 2:
    resid_df = pd.DataFrame(resids).dropna()
    resid_corr = resid_df.corr(method='spearman')
    print('Residual Spearman correlations between metals:')
    print(resid_corr.round(3))

    try:
        import seaborn as sns
        fig, ax = plt.subplots(figsize=(4, 4))
        sns.heatmap(resid_corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
                    xticklabels=[t.replace('log_', '') for t in resid_corr.columns],
                    yticklabels=[t.replace('log_', '') for t in resid_corr.index], ax=ax)
        ax.set_title('Residual correlations (M2)')
        plt.tight_layout()
        plt.savefig(FIGURES_DIR / 'metal_residual_correlations.png', dpi=150)
        plt.close()
        print('Metal residual correlation heatmap saved.')
    except ImportError:
        print('seaborn not available — skipping heatmap')
print('=== NB04 COMPLETE ===')